In [188]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from qa_data import ml_qa_data
import re
from textblob import TextBlob
from nltk.corpus import stopwords, wordnet

In [189]:
questions = list(ml_qa_data.keys())

In [190]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = text.strip()
    text = str(TextBlob(text).correct())
    
    return text.strip()

In [191]:
cleaned_question = [preprocess(q) for q in questions]

In [192]:
def get_synonyms(word):
    synonyms = set()

    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name())
    return " ".join(synonyms)

In [193]:
print(get_synonyms("ML"))

millilitre cubic_centimeter cc milliliter cubic_centimetre mil ml


In [194]:
full_mean = {
    "ai": "artificial intelligence",
    "ml": "machine learning",
    "pca": "principal component analysis",
    "svm": "support vector machine",
    "knn": "k nearest neighbors",
    "f1": "f1 score",
    "l1": "lasso regularization",
    "l2": "ridge regularization",
    "xgboost": "extreme gradient boosting"
}

def use_full_mean(text):
    words = text.lower().split()
    expanded = []

    for w in words:
        if w in full_mean:
            expanded.append(full_mean[w])
        else:
            expanded.append(w)
    return " ".join(expanded)

In [195]:
use_full_mean('ML')

'machine learning'

In [196]:
tfidf = TfidfVectorizer(encoding='utf-8', lowercase=True)

In [197]:
vect_data = tfidf.fit_transform(cleaned_question)

In [198]:
def get_faq_quetion(user_input: str, threshold = .4) -> str:

    query = preprocess(user_input)
    query = use_full_mean(query)
    query_vec = tfidf.transform([query])
    similarity = cosine_similarity(query_vec, vect_data).flatten()
    q_match = np.argmax(similarity)

    score = similarity[q_match]
    print(f'score: {score}')
    
    if score > threshold:
        answer = questions[q_match]
        return ml_qa_data[answer]
    else:
        return "Please! Ask related Question!"

In [204]:
get_faq_quetion("what is svm?")

score: 0.3592179240846274


'Please! Ask related Question!'